# Figure 5 — Plagioclase and orthopyroxene compositions and zoning


## 1. Imports and shared 00_Setup

This notebook uses Arial, 8-point ordinary text, 9-point bold uppercase panel labels, full axes boxes, light gridlines, 300 dpi PNG export, and editable PDF/SVG text. The code is linear and contains no custom functions.


In [1]:
from pathlib import Path
import hashlib
import re

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from PIL import Image

SINGLE_COLUMN_WIDTH_IN = 3.55
DOUBLE_COLUMN_WIDTH_IN = 7.25
MAXIMUM_FIGURE_HEIGHT_IN = 7.8
FIGURE_WIDTH_IN = DOUBLE_COLUMN_WIDTH_IN
FIGURE_HEIGHT_IN = 7.6
BASE_TEXT_SIZE_PT = 8.0
PANEL_LABEL_SIZE_PT = 9.0
LEGEND_SIZE_PT = 6.0
MINIMUM_LINE_WIDTH_PT = 0.28
EXPORT_DPI = 300
PUBLICATION_FONT = "Arial"

installed_font_names = {entry.name for entry in font_manager.fontManager.ttflist}
if PUBLICATION_FONT not in installed_font_names:
    raise RuntimeError("Arial is required by 00_Setup.")
mpl.rcParams.update({
    "font.family":"sans-serif", "font.sans-serif":[PUBLICATION_FONT],
    "font.size":BASE_TEXT_SIZE_PT, "axes.labelsize":BASE_TEXT_SIZE_PT,
    "axes.titlesize":BASE_TEXT_SIZE_PT, "xtick.labelsize":BASE_TEXT_SIZE_PT,
    "ytick.labelsize":BASE_TEXT_SIZE_PT, "legend.fontsize":LEGEND_SIZE_PT,
    "figure.facecolor":"white", "axes.facecolor":"white",
    "axes.spines.top":True, "axes.spines.right":True, "axes.linewidth":0.5,
    "lines.linewidth":0.75, "xtick.direction":"in", "ytick.direction":"in",
    "xtick.major.width":0.5, "ytick.major.width":0.5,
    "savefig.dpi":EXPORT_DPI, "savefig.facecolor":"white",
    "savefig.transparent":False, "pdf.fonttype":42, "ps.fonttype":42,
    "svg.fonttype":"none",
})


## 2. Paths and public S1–S3 workbooks

Figure 5 uses the public S1 transects, melt-inclusion hosts, and mingled-mineral sheets. S2 and S3 are checked as part of the shared package but are not used for the plotted populations.


In [2]:
PROJECT_ROOT_OVERRIDE = None
PROJECT_ROOT = PROJECT_ROOT_OVERRIDE
if PROJECT_ROOT is None:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "SA_MI_Draft_FinalFigures_Work" / "DataSheets").is_dir():
            PROJECT_ROOT = candidate
            break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project or set PROJECT_ROOT_OVERRIDE.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
WORK_ROOT = PROJECT_ROOT / "SA_MI_Draft_FinalFigures_Work"
DATA_DIRECTORY = WORK_ROOT / "DataSheets" / "Updated_Oct20"
FIGURE_DIRECTORY = WORK_ROOT / "Nov6_AllFigures" / "Figure_05_Transects_Histograms"
OUTPUT_DIRECTORY = FIGURE_DIRECTORY / "Figure_05_Transects_Histograms_Final"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

S1_PATH = DATA_DIRECTORY / "Dechert_S1_AllData.xlsx"
S2_PATH = DATA_DIRECTORY / "Dechert_S2_Standards.xlsx"
S3_PATH = DATA_DIRECTORY / "Dechert_S3_Compilations.xlsx"
for workbook_path in [S1_PATH, S2_PATH, S3_PATH]:
    if not workbook_path.is_file():
        raise FileNotFoundError(f"Missing supplemental workbook: {workbook_path}")
SOURCE_HASHES_BEFORE = {
    "S1":hashlib.sha256(S1_PATH.read_bytes()).hexdigest(),
    "S2":hashlib.sha256(S2_PATH.read_bytes()).hexdigest(),
    "S3":hashlib.sha256(S3_PATH.read_bytes()).hexdigest(),
}


## 3. Read the established S1 populations and configure the two minerals


In [3]:
transects_source = pd.read_excel(S1_PATH, sheet_name="Transects")
hosts_source = pd.read_excel(S1_PATH, sheet_name="MI_Hosts_Plag_Pxy")
mingled_source = pd.read_excel(S1_PATH, sheet_name="Mingled_Minerals")

all_linescans = transects_source.rename(columns={"Sample":"Sample_Group"}).copy()
scan_counts = all_linescans.groupby("Linescan_ID")["Predict_Mineral"].value_counts(dropna=False)
scan_total = all_linescans.groupby("Linescan_ID").size()
dominant_table = (
    scan_counts.rename("n").reset_index()
    .sort_values(["Linescan_ID","n"], ascending=[True,False], kind="stable")
    .drop_duplicates("Linescan_ID")
)
dominant = dominant_table.set_index("Linescan_ID")["Predict_Mineral"]
fraction = scan_counts.groupby(level=0).max().div(scan_total)
all_linescans["Dominant_Mineral"] = all_linescans["Linescan_ID"].map(dominant)
all_linescans["Dominant_Mineral_Fraction"] = all_linescans["Linescan_ID"].map(fraction)
assert all_linescans["Sample_Group"].isin(["74A","74D","75","76A"]).all()
assert fraction.ge(0.95 - 1e-12).all()
assert not all_linescans["Source_CSV"].astype("string").str.contains("75_1_C32|74A2_amph9|^76B", case=False, regex=True, na=False).any()
mingled_minerals = mingled_source.rename(columns={"Type of Pumice":"Mafic/Felsic"}).copy()
dechert_combined_data = hosts_source.copy()

PAPER_POOL_MEMBERS = {"B":["74D","74A"], "G":["75","76A"]}
PAPER_POOL_ORDER = ["B","G"]
PAPER_LINE_CMAPS = {"B":"Oranges", "G":"Blues"}
PAPER_MAIN_COLORS = {
    "B":{"fill":"#F4A261","outline":"#D55E00"},
    "G":{"fill":"#56B4E9","outline":"#0072B2"},
}
PAPER_MINGLED_COLORS = {
    "B":{"light":"#EE3377","dark":"#808080"},
    "G":{"light":"#66C2C2","dark":"#000000"},
}
PAPER_MINGLED_DATASET = {"B":"74Ming", "G":"76Ming"}
MINGLED_DISPLAY_LABELS = {"Felsic":"light","Mafic":"dark","light":"light","dark":"dark"}
MINERAL_CONFIGURATION = {
    "Plagioclase":{
        "source_phase":"Plagioclase", "value_column":"An", "host_phase":"plag",
        "host_prediction":"Plagioclase", "host_value_column":"An#_host",
        "axis_label":"An", "title":"Plagioclase Transects", "row_offset":0,
    },
    "Orthopyroxene":{
        "source_phase":"Orthopyroxene", "value_column":"Mg#", "host_phase":"pxy",
        "host_prediction":"Orthopyroxene", "host_value_column":"Mg#_host",
        "axis_label":"Mg#", "title":"Orthopyroxene Transects", "row_offset":2,
    },
}
MML_PREDICTION_SCORE_MIN = 0.85
MINERAL_DOMINANCE_MIN = 0.95
HIST_OVERSAMPLE_MULTIPLIER = 10
N_HISTOGRAM_EDGES = 20


## 4. Select transects, resample to equal weight, and collect hosts and mingled pumice


In [4]:
linescan_work = all_linescans.copy()
linescan_work["_distance_um"] = pd.to_numeric(linescan_work["Distance (µm)"], errors="coerce")
linescan_work["_prediction_score"] = pd.to_numeric(linescan_work["Prediction_Score"], errors="coerce")
linescan_work["_dominant_fraction"] = pd.to_numeric(linescan_work["Dominant_Mineral_Fraction"], errors="coerce")
standard_pattern = r"(?:albite|kl_?2g?|macfo|anorthite|labradorite|lab\d)"
linescan_work["_is_standard"] = linescan_work["Source_CSV"].astype("string").str.contains(standard_pattern, case=False, regex=True, na=False)
linescan_work["_is_discarded"] = False
for discard_column in ["discard","Discard"]:
    if discard_column in linescan_work.columns:
        linescan_work["_is_discarded"] = linescan_work["_is_discarded"] | linescan_work[discard_column].astype("string").str.strip().str.casefold().isin(["1","true","yes","y","discard","discarded"]).fillna(False)
nonstandard_nondiscarded = ~linescan_work["_is_standard"] & ~linescan_work["_is_discarded"]
base_linescan_mask = nonstandard_nondiscarded & linescan_work["Sample_Group"].isin(["74A","74D","75","76A"])
HIST_MAX_ORIGINAL_POINTS = int(linescan_work.loc[nonstandard_nondiscarded].groupby("Linescan_ID")["_distance_um"].count().max())
HIST_N_RESAMPLE = HIST_MAX_ORIGINAL_POINTS * HIST_OVERSAMPLE_MULTIPLIER

LINE_SCAN_COHORTS = {}
EQUAL_WEIGHT_TRANSECT_VALUES = {}
for mineral, configuration in MINERAL_CONFIGURATION.items():
    numeric_values = pd.to_numeric(linescan_work[configuration["value_column"]], errors="coerce")
    selected_mask = (
        base_linescan_mask
        & linescan_work["Dominant_Mineral"].astype("string").eq(configuration["source_phase"])
        & linescan_work["_dominant_fraction"].ge(MINERAL_DOMINANCE_MIN)
        & linescan_work["Predict_Mineral"].astype("string").eq(configuration["source_phase"])
        & linescan_work["_prediction_score"].gt(MML_PREDICTION_SCORE_MIN)
        & linescan_work["_distance_um"].notna()
        & numeric_values.notna()
    )
    selected = linescan_work.loc[selected_mask].copy()
    selected["_plot_value"] = numeric_values.loc[selected_mask]
    LINE_SCAN_COHORTS[mineral] = {}
    EQUAL_WEIGHT_TRANSECT_VALUES[mineral] = {}
    for pool in PAPER_POOL_ORDER:
        pool_transects = []
        for sample in PAPER_POOL_MEMBERS[pool]:
            sample_frame = selected.loc[selected["Sample_Group"].eq(sample)]
            for linescan_id, transect in sample_frame.groupby("Linescan_ID", sort=True):
                plotted = transect.sort_values(["_distance_um","Point_Order"], kind="stable").copy()
                if len(plotted) == 0:
                    continue
                LINE_SCAN_COHORTS[mineral].setdefault(pool, []).append(plotted)
        LINE_SCAN_COHORTS[mineral].setdefault(pool, [])
        interpolated = []
        for transect in LINE_SCAN_COHORTS[mineral][pool]:
            xy = transect[["_distance_um","_plot_value"]].dropna().groupby("_distance_um", as_index=False)["_plot_value"].mean().sort_values("_distance_um")
            if len(xy) == 1:
                interpolated.append(np.repeat(float(xy["_plot_value"].iloc[0]), HIST_N_RESAMPLE))
            elif len(xy) > 1:
                interpolated.append(np.interp(
                    np.linspace(0.0, 1.0, HIST_N_RESAMPLE),
                    np.linspace(0.0, 1.0, len(xy)),
                    xy["_plot_value"].to_numpy(float),
                ))
        EQUAL_WEIGHT_TRANSECT_VALUES[mineral][pool] = np.concatenate(interpolated) if interpolated else np.array([], dtype=float)

host_work = dechert_combined_data.copy()
host_work["_sample_group"] = pd.NA
for sample in ["74D","74A","75","76A"]:
    host_work.loc[host_work["base_name"].astype("string").str.startswith(sample, na=False), "_sample_group"] = sample
host_work["_prediction_score"] = pd.to_numeric(host_work["Prediction_Score_host"], errors="coerce")
HOST_MI_VALUES = {}
for mineral, configuration in MINERAL_CONFIGURATION.items():
    numeric_values = pd.to_numeric(host_work[configuration["host_value_column"]], errors="coerce")
    host_mask = (
        host_work["mineral_phase"].astype("string").eq(configuration["host_phase"])
        & host_work["Predict_Mineral_host"].astype("string").eq(configuration["host_prediction"])
        & host_work["_prediction_score"].gt(MML_PREDICTION_SCORE_MIN)
        & numeric_values.notna()
    )
    HOST_MI_VALUES[mineral] = {
        pool:numeric_values.loc[host_mask & host_work["_sample_group"].isin(PAPER_POOL_MEMBERS[pool])].to_numpy(float)
        for pool in PAPER_POOL_ORDER
    }

mingled_work = mingled_minerals.copy()
mingled_work["_prediction_score"] = pd.to_numeric(mingled_work["Prediction_Score"], errors="coerce")
mingled_work["_display_texture"] = mingled_work["Mafic/Felsic"].map(MINGLED_DISPLAY_LABELS)
mingled_work["_display_texture"] = mingled_work["_display_texture"].fillna(
    mingled_work["Mafic/Felsic"].astype("string").str.strip().str.casefold().map(MINGLED_DISPLAY_LABELS)
)
MINGLED_VALUES = {}
for mineral, configuration in MINERAL_CONFIGURATION.items():
    numeric_values = pd.to_numeric(mingled_work[configuration["value_column"]], errors="coerce")
    mineral_mask = (
        mingled_work["Predict_Mineral"].astype("string").eq(configuration["source_phase"])
        & mingled_work["_prediction_score"].gt(MML_PREDICTION_SCORE_MIN)
        & numeric_values.notna()
    )
    MINGLED_VALUES[mineral] = {
        pool:{
            texture:numeric_values.loc[
                mineral_mask
                & mingled_work["Dataset"].astype("string").eq(PAPER_MINGLED_DATASET[pool])
                & mingled_work["_display_texture"].eq(texture)
            ].to_numpy(float)
            for texture in ["light","dark"]
        }
        for pool in PAPER_POOL_ORDER
    }

assert all(len(HOST_MI_VALUES[mineral][pool]) > 0 for mineral in MINERAL_CONFIGURATION for pool in PAPER_POOL_ORDER)
assert all(len(MINGLED_VALUES[mineral][pool][texture]) > 0 for mineral in MINERAL_CONFIGURATION for pool in PAPER_POOL_ORDER for texture in ["light","dark"])


## 5. Set histogram ranges and build the 4×3 figure


In [5]:
HISTOGRAM_BINS = {}
HISTOGRAM_X_LIMITS = {}
MINERAL_Y_LIMITS = {}
FIGURE_5_AUDIT = []
for mineral, configuration in MINERAL_CONFIGURATION.items():
    displayed_arrays = []
    for pool in PAPER_POOL_ORDER:
        displayed_arrays.extend([
            EQUAL_WEIGHT_TRANSECT_VALUES[mineral][pool],
            HOST_MI_VALUES[mineral][pool],
            MINGLED_VALUES[mineral][pool]["light"],
            MINGLED_VALUES[mineral][pool]["dark"],
        ])
    displayed_arrays = [values for values in displayed_arrays if len(values)]
    axis_min = min(float(np.nanmin(values)) for values in displayed_arrays)
    axis_max = max(float(np.nanmax(values)) for values in displayed_arrays)
    padding = (axis_max - axis_min) * 0.025 if axis_max != axis_min else 1.0
    MINERAL_Y_LIMITS[mineral] = (axis_min - padding, axis_max + padding)
    HISTOGRAM_BINS[mineral] = np.linspace(axis_min, axis_max, N_HISTOGRAM_EDGES)
    density_maxima = []
    for values in displayed_arrays:
        density, _ = np.histogram(values, bins=HISTOGRAM_BINS[mineral], density=True)
        density_maxima.append(float(np.nanmax(density)))
    HISTOGRAM_X_LIMITS[mineral] = max(density_maxima) * 1.08
    for pool in PAPER_POOL_ORDER:
        transect_frames = LINE_SCAN_COHORTS[mineral][pool]
        transect_values = EQUAL_WEIGHT_TRANSECT_VALUES[mineral][pool]
        host_values = HOST_MI_VALUES[mineral][pool]
        light_values = MINGLED_VALUES[mineral][pool]["light"]
        dark_values = MINGLED_VALUES[mineral][pool]["dark"]
        FIGURE_5_AUDIT.append({
            "mineral":mineral, "pool":pool, "n_transects":len(transect_frames),
            "n_linescan_points":sum(len(frame) for frame in transect_frames),
            "n_transect_hist_values":len(transect_values),
            "n_host_mi_values":len(host_values), "n_mingled_light_values":len(light_values),
            "n_mingled_dark_values":len(dark_values),
        })

figure = plt.figure(figsize=(FIGURE_WIDTH_IN, FIGURE_HEIGHT_IN))
grid = figure.add_gridspec(5, 3, width_ratios=(4.10,1.45,1.45), height_ratios=(1.0,1.0,0.16,1.0,1.0), wspace=0.10, hspace=0.10)
axes = np.empty((4,3), dtype=object)
panel_number = 0

for mineral, configuration in MINERAL_CONFIGURATION.items():
    bins = HISTOGRAM_BINS[mineral]
    y_limits = MINERAL_Y_LIMITS[mineral]
    histogram_x_limit = HISTOGRAM_X_LIMITS[mineral]
    y_ticks = np.arange(30,81,10) if mineral == "Plagioclase" else np.arange(60,81,4)
    for pool_index, pool in enumerate(PAPER_POOL_ORDER):
        row = configuration["row_offset"] + pool_index
        plot_grid_row = row if row < 2 else row + 1
        ax_line = figure.add_subplot(grid[plot_grid_row,0])
        ax_main = figure.add_subplot(grid[plot_grid_row,1], sharey=ax_line)
        ax_mingled = figure.add_subplot(grid[plot_grid_row,2], sharey=ax_line)
        axes[row] = (ax_line, ax_main, ax_mingled)
        transect_frames = LINE_SCAN_COHORTS[mineral][pool]
        line_cmap = plt.colormaps[PAPER_LINE_CMAPS[pool]]
        color_positions = np.linspace(0.46,0.92,max(len(transect_frames),2))
        for index, transect in enumerate(transect_frames):
            ax_line.plot(transect["_distance_um"].to_numpy(float), transect["_plot_value"].to_numpy(float), "-o", color=line_cmap(color_positions[index]), linewidth=0.85, markersize=1.80, markeredgewidth=0.0, alpha=0.90, zorder=2)
        transect_values = EQUAL_WEIGHT_TRANSECT_VALUES[mineral][pool]
        host_values = HOST_MI_VALUES[mineral][pool]
        fill_color = PAPER_MAIN_COLORS[pool]["fill"]
        outline_color = PAPER_MAIN_COLORS[pool]["outline"]
        ax_main.hist(host_values, bins=bins, density=True, orientation="horizontal", color=fill_color, edgecolor=fill_color, alpha=0.55, linewidth=0.65)
        ax_main.hist(transect_values, bins=bins, density=True, orientation="horizontal", histtype="step", color=outline_color, linewidth=0.90)
        host_median = float(np.median(host_values))
        transect_median = float(np.median(transect_values))
        ax_main.axhline(host_median, color=fill_color, linestyle="--", linewidth=0.75)
        ax_main.axhline(transect_median, color=outline_color, linestyle="--", linewidth=0.75)
        ax_main.legend(handles=[
            Patch(facecolor=fill_color, edgecolor=fill_color, alpha=0.55, label=f"MI host n={len(host_values)}\nmed={host_median:.0f}"),
            Patch(facecolor="none", edgecolor=outline_color, linewidth=0.90, label=f"tran. n={len(transect_frames)}\nmed={transect_median:.0f}"),
        ], loc="upper right", fontsize=6.0, frameon=False, borderpad=0.0, labelspacing=0.18, handlelength=1.05, handletextpad=0.35)
        light_values = MINGLED_VALUES[mineral][pool]["light"]
        dark_values = MINGLED_VALUES[mineral][pool]["dark"]
        light_color = PAPER_MINGLED_COLORS[pool]["light"]
        dark_color = PAPER_MINGLED_COLORS[pool]["dark"]
        ax_mingled.hist(light_values, bins=bins, density=True, orientation="horizontal", color=light_color, edgecolor=light_color, alpha=0.58, linewidth=0.65)
        ax_mingled.hist(dark_values, bins=bins, density=True, orientation="horizontal", histtype="step", color=dark_color, linewidth=0.90)
        light_median = float(np.median(light_values))
        dark_median = float(np.median(dark_values))
        ax_mingled.axhline(light_median, color=light_color, linestyle="--", linewidth=0.75)
        ax_mingled.axhline(dark_median, color=dark_color, linestyle=":", linewidth=0.90)
        phase_suffix = " Opx" if mineral == "Orthopyroxene" else ""
        ax_mingled.legend(handles=[
            Patch(facecolor=light_color, edgecolor=light_color, alpha=0.58, label=f"light{phase_suffix} n={len(light_values)}\nmed={light_median:.0f}"),
            Patch(facecolor="none", edgecolor=dark_color, linewidth=0.90, label=f"dark{phase_suffix} n={len(dark_values)}\nmed={dark_median:.0f}"),
        ], loc="upper right", fontsize=6.0, frameon=False, borderpad=0.0, labelspacing=0.18, handlelength=1.05, handletextpad=0.35)
        for axis in [ax_line, ax_main, ax_mingled]:
            axis.set_ylim(*y_limits)
            axis.grid(True, axis="y", color="0.92", linewidth=0.35, zorder=0)
            axis.set_axisbelow(True)
            for spine in axis.spines.values():
                spine.set_visible(True)
                spine.set_color("black")
                spine.set_linewidth(0.50)
            panel_number += 1
            axis.text(0.018,0.982,f"{chr(64+panel_number)})", transform=axis.transAxes, ha="left", va="top", fontsize=PANEL_LABEL_SIZE_PT, fontweight="bold", bbox={"facecolor":"white","edgecolor":"none","alpha":0.82,"pad":0.35}, zorder=20)
        ax_line.set_xlim(left=0)
        ax_line.set_ylabel(configuration["axis_label"])
        ax_line.set_yticks(y_ticks)
        ax_line.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%.0f"))
        ax_line.text(0.985,0.955,f"Sample {pool}\n{len(transect_frames)} transects; {sum(len(frame) for frame in transect_frames):,} points", transform=ax_line.transAxes, ha="right", va="top", fontsize=6.0, bbox={"facecolor":"white","edgecolor":"none","alpha":0.78,"pad":0.8}, zorder=10)
        ax_line.ticklabel_format(axis="x", style="plain", useOffset=False)
        for histogram_axis in [ax_main, ax_mingled]:
            histogram_axis.set_xlim(0, histogram_x_limit)
            histogram_axis.set_yticks(y_ticks)
            histogram_axis.tick_params(axis="y", left=True, right=False, labelleft=False, labelright=False, length=2.5, width=0.5)
            histogram_axis.tick_params(axis="x", bottom=False, top=False, labelbottom=False, labeltop=False)
        if pool_index == 1:
            ax_line.set_xlabel("Distance (µm)")
            x_min, x_max = ax_line.get_xlim()
            for label, x_position in [("Rim",x_min),("Core",x_max)]:
                ax_line.annotate(label, xy=(x_position,0), xycoords=("data","axes fraction"), xytext=(0,-17), textcoords="offset points", ha="center", va="top", fontsize=6.0, fontstyle="italic")
        else:
            ax_line.tick_params(axis="x", labelbottom=False)
        if row == 0:
            ax_line.set_title("Plagioclase Transects", pad=4)
            ax_main.set_title("Main pumice", pad=4)
            ax_mingled.set_title("Mingled pumice", pad=4)
        elif row == 2:
            ax_line.set_title("Orthopyroxene Transects", pad=4)

figure.subplots_adjust(left=0.075, right=0.995, bottom=0.065, top=0.965)
FIGURE_STEM = "Figure_05_Transects_Histograms"
EXPORT_PATHS = {}
for output_format in ["pdf","png","svg"]:
    output_path = OUTPUT_DIRECTORY / f"{FIGURE_STEM}.{output_format}"
    save_options = {"format":output_format, "facecolor":"white", "transparent":False}
    if output_format == "png":
        save_options["dpi"] = EXPORT_DPI
    figure.savefig(output_path, **save_options)
    if not output_path.is_file() or output_path.stat().st_size == 0:
        raise IOError(f"Expected export was not created: {output_path}")
    EXPORT_PATHS[output_format] = output_path
plt.close(figure)


## 6. Validate the cohorts, label changes, histogram ticks, and exports


In [6]:
FIGURE_5_AUDIT = pd.DataFrame(FIGURE_5_AUDIT)
assert FIGURE_5_AUDIT["n_transects"].gt(0).all()
assert FIGURE_5_AUDIT["n_linescan_points"].gt(0).all()
assert FIGURE_5_AUDIT["n_transect_hist_values"].eq(FIGURE_5_AUDIT["n_transects"] * HIST_N_RESAMPLE).all()
assert MINERAL_CONFIGURATION["Plagioclase"]["axis_label"] == "An"
assert MINERAL_CONFIGURATION["Orthopyroxene"]["axis_label"] == "Mg#"
assert panel_number == 12
assert tuple(round(value, 8) for value in figure.get_size_inches()) == (7.25, 7.6)
with Image.open(EXPORT_PATHS["png"]) as image:
    assert image.size == (2175, 2280)
svg_text = EXPORT_PATHS["svg"].read_text(encoding="utf-8", errors="replace")
assert "<text" in svg_text and "<image" not in svg_text
assert b"/Type3" not in EXPORT_PATHS["pdf"].read_bytes()
SOURCE_HASHES_AFTER = {
    "S1":hashlib.sha256(S1_PATH.read_bytes()).hexdigest(),
    "S2":hashlib.sha256(S2_PATH.read_bytes()).hexdigest(),
    "S3":hashlib.sha256(S3_PATH.read_bytes()).hexdigest(),
}
assert SOURCE_HASHES_AFTER == SOURCE_HASHES_BEFORE
print("Figure 5 complete.")
print("Outputs:", {key:str(value) for key,value in EXPORT_PATHS.items()})


Figure 5 complete.
Outputs: {'pdf': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_05_Transects_Histograms/Figure_05_Transects_Histograms_Final/Figure_05_Transects_Histograms.pdf', 'png': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_05_Transects_Histograms/Figure_05_Transects_Histograms_Final/Figure_05_Transects_Histograms.png', 'svg': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_05_Transects_Histograms/Figure_05_Transects_Histograms_Final/Figure_05_Transects_Histograms.svg'}
